# Module 6.1: Automated Local Nightly Backups (`homelab-pi` + `homelab-gpu` $\rightarrow$ `UNAS 2`) & Immich External Library

## UniFi Storage & Enterprise Networking (Chapter 3) - Step 6: Automated 3-2-1 Cloud Backups & Observability

## 🎯 Learning Goals

By completing this laboratory module, you will be able to:

1. Enforce the **Tiered Storage Golden Rule**: keep live PostgreSQL databases (`Immich`, `Paperless-ngx`, `Mail-Archiver`) and MongoDB (`UniFi Network`) on fast local NVMe SSDs—never directly over NFS—while streaming crash-consistent nightly dumps onto `UNAS 2` RAID 1.
2. Deploy an automated nightly **`systemd` timer** on **`homelab-pi`** (`03:00 AM`) that performs a live `pg_dumpall` of Immich's PostgreSQL database and incrementally mirrors your Immich photo/video originals, Pi-hole v6 configs, and Uptime Kuma data onto `/mnt/unas_backups/homelab-pi/`.
3. Deploy a comprehensive nightly **`systemd` timer** on **`homelab-gpu`** (`03:30 AM`) that dumps **Paperless-ngx** (database + `document_exporter`), **Mail-Archiver** (`pg_dump` of archived emails), **UniFi Network** `.unf` auto-backups, and all Docker Compose stacks onto `/mnt/unas_backups/homelab-gpu/`.
4. Optionally attach `UNAS 2`'s `/mnt/unas_media` share read-only into `immich-server` as an **Immich External Library** so legacy photo archives from your University OneDrive appear seamlessly inside Immich without consuming Pi NVMe space.

## 6.1.1 Conceptual Background: Unified Local Backup Architecture Across All Three Nodes

Your homelab now hosts **four stateful databases** plus terabytes of irreplaceable photos, migrated university archives, OCR documents, and archived emails across two compute nodes (`homelab-pi` and `homelab-gpu`). Before we encrypt and push everything offsite to your **5 TB Google Drive** in Module 6.2, we consolidate **every single byte of critical homelab state** into one clean, RAID-1 protected, Btrfs-snapshotted staging hub on **`UNAS 2` (`10.0.0.202`)**!

```mermaid
flowchart LR
    subgraph Pi ["Node 1: homelab-pi (10.0.0.200) | 03:00 AM Timer"]
        PiDB["Immich Postgres<br>(pg_dumpall -> .sql.gz)"]
        PiLib["Immich Originals<br>+ Pi-hole + Uptime Kuma"]
    end
    
    subgraph GPU ["Node 2: homelab-gpu (10.0.0.201) | 03:30 AM Timer"]
        PngxDB["Paperless-ngx Postgres<br>(pg_dump -> .sql.gz)"]
        MailDB["Mail-Archiver Postgres<br>(pg_dump -> .sql.gz)"]
        UniFiBak["UniFi Network .unf Backups<br>+ All Compose Stacks"]
    end
    
    subgraph UNAS ["Node 3: UniFi UNAS 2 (10.0.0.202 | 2x 4TB RAID 1 + Btrfs Snapshots)"]
        HB["/HomelabBackups<br>(Consolidated Nightly DR Target)"]
        MA["/MediaArchive<br>(University Archive + Paperless PDFs + Personal Files)"]
    end
    
    Pi ==>|"NFSv3 over 2.5GbE Switch"| HB
    GPU ==>|"NFSv3 over 2.5GbE Switch"| HB
    HB -. "Module 6.2 (04:30 AM):<br>rclone crypt Zero-Knowledge Sync" .-> GD["5 TB Google Drive<br>(Included w/ Gemini Pro)"]
    MA -. "Module 6.2 (04:30 AM):<br>rclone crypt Zero-Knowledge Sync" .-> GD
```

### Why Dump Every PostgreSQL Database with `pg_dump` Instead of Raw `rsync`?
Copying a live `/var/lib/postgresql/data` directory with `rsync` while Postgres is actively writing WAL segments captures an inconsistent half-written snapshot that can fail to boot during disaster recovery. Running `docker exec <pg_container> pg_dump` (or `pg_dumpall`) produces a **100% transactionally consistent, compressed `.sql.gz`** archive with zero container downtime!

## 6.1.2 Step-by-Step Laboratory Guide

### Task 6.1.1: Automate Nightly `homelab-pi` Backups to `UNAS 2` (`03:00 AM` `systemd` Timer)
1. SSH into `homelab-pi` (`ssh homelab-pi`) and create `/usr/local/bin/homelab-backup.sh`:
   ```bash
   sudo tee /usr/local/bin/homelab-backup.sh > /dev/null << 'EOF'
   #!/usr/bin/env bash
   set -euo pipefail

   BACKUP_ROOT="/mnt/unas_backups/homelab-pi"
   DATE=$(date +%F_%H%M)
   LOG_TAG="homelab-pi-backup"

   logger -t "$LOG_TAG" "Starting nightly backup from homelab-pi to UNAS 2..."

   # Trigger systemd automount & abort safely if UNAS 2 is unreachable
   ls /mnt/unas_backups >/dev/null 2>&1 || true
   if ! mountpoint -q /mnt/unas_backups; then
       logger -t "$LOG_TAG" "ERROR: /mnt/unas_backups is NOT mounted! Aborting."
       exit 1
   fi

   mkdir -p "$BACKUP_ROOT"/{immich-db,immich-library,configs}

   # 1. Crash-consistent Immich PostgreSQL dump (keep 14 days)
   if docker ps --format '{{.Names}}' | grep -q '^immich_postgres$'; then
       docker exec immich_postgres pg_dumpall --clean --if-exists --username=postgres \
           | gzip > "$BACKUP_ROOT/immich-db/immich_db_${DATE}.sql.gz"
       find "$BACKUP_ROOT/immich-db" -name "immich_db_*.sql.gz" -mtime +14 -delete
   fi

   # 2. Incremental mirror of Immich original photos & videos (skip regenerable thumbs/encoded cache)
   if [ -d /mnt/homelab_storage/immich_library ]; then
       rsync -a --delete --no-perms --no-owner --no-group \
           --exclude='thumbs/' --exclude='encoded-video/' \
           /mnt/homelab_storage/immich_library/ \
           "$BACKUP_ROOT/immich-library/"
   fi

   # 3. Mirror Docker Compose stacks, Pi-hole v6 configs, and Uptime Kuma data
   rsync -a --no-perms --no-owner --no-group \
       /home/pi/homelab/ \
       "$BACKUP_ROOT/configs/homelab-stacks/"

   if [ -d /mnt/homelab_storage/docker_volumes ]; then
       rsync -a --no-perms --no-owner --no-group \
           /mnt/homelab_storage/docker_volumes/ \
           "$BACKUP_ROOT/configs/docker-volumes/"
   fi

   logger -t "$LOG_TAG" "Nightly homelab-pi backup completed successfully!"
   EOF
   sudo chmod +x /usr/local/bin/homelab-backup.sh
   ```
2. Install and enable the daily **03:00 AM** `systemd` timer:
   ```bash
   sudo tee /etc/systemd/system/homelab-backup.service > /dev/null << 'EOF'
   [Unit]
   Description=Nightly Homelab-Pi Backup to UniFi UNAS 2
   After=network-online.target docker.service
   Wants=network-online.target

   [Service]
   Type=oneshot
   ExecStart=/usr/local/bin/homelab-backup.sh
   EOF

   sudo tee /etc/systemd/system/homelab-backup.timer > /dev/null << 'EOF'
   [Unit]
   Description=Run Homelab-Pi Backup Daily at 03:00 AM

   [Timer]
   OnCalendar=*-*-* 03:00:00
   RandomizedDelaySec=10m
   Persistent=true

   [Install]
   WantedBy=timers.target
   EOF

   sudo systemctl daemon-reload
   sudo systemctl enable --now homelab-backup.timer
   sudo systemctl start homelab-backup.service
   journalctl -u homelab-backup.service -n 15 --no-pager
   ```

### Task 6.1.2: Automate Nightly `homelab-gpu` Backups (Paperless-ngx, Mail-Archiver & UniFi Network)
Now create the matching **03:30 AM** backup pipeline on `homelab-gpu` (`ssh homelab-gpu`) so your OCR database, archived `.edu`/Gmail emails, and UniFi Network switch configs are all backed up cleanly onto `UNAS 2` before Module 6.2 pushes everything to Google Drive!

```bash
sudo tee /usr/local/bin/homelab-gpu-backup.sh > /dev/null << 'EOF'
#!/usr/bin/env bash
set -euo pipefail

BACKUP_ROOT="/mnt/unas_backups/homelab-gpu"
DATE=$(date +%F_%H%M)
LOG_TAG="homelab-gpu-backup"
USER_HOME="/home/pi"

logger -t "$LOG_TAG" "Starting nightly backup from homelab-gpu to UNAS 2..."

ls /mnt/unas_backups >/dev/null 2>&1 || true
if ! mountpoint -q /mnt/unas_backups; then
    logger -t "$LOG_TAG" "ERROR: /mnt/unas_backups is NOT mounted! Aborting."
    exit 1
fi

mkdir -p "$BACKUP_ROOT"/{paperless-db,mailarchiver-db,unifi-autobackups,homelab-stacks}

# 1. Paperless-ngx PostgreSQL dump (keep 14 days)
if docker ps --format '{{.Names}}' | grep -q '^paperless_db$'; then
    docker exec paperless_db pg_dump -U paperless paperless \
        | gzip > "$BACKUP_ROOT/paperless-db/paperless_db_${DATE}.sql.gz"
    find "$BACKUP_ROOT/paperless-db" -name "paperless_db_*.sql.gz" -mtime +14 -delete
fi

# 2. Mail-Archiver PostgreSQL dump (contains all archived .edu & Gmail emails + attachments! Keep 14 days)
if docker ps --format '{{.Names}}' | grep -q '^mailarchiver_postgres$'; then
    docker exec mailarchiver_postgres pg_dump -U mailuser MailArchiver \
        | gzip > "$BACKUP_ROOT/mailarchiver-db/mailarchiver_db_${DATE}.sql.gz"
    find "$BACKUP_ROOT/mailarchiver-db" -name "mailarchiver_db_*.sql.gz" -mtime +14 -delete
fi

# 3. UniFi Network Application built-in .unf auto-backups
if [ -d "$USER_HOME/homelab/stacks/unifi-network/config/data/backup" ]; then
    rsync -a --no-perms --no-owner --no-group \
        "$USER_HOME/homelab/stacks/unifi-network/config/data/backup/" \
        "$BACKUP_ROOT/unifi-autobackups/"
fi

# 4. All Docker Compose YAMLs, .env files, FileBrowser config & DataProtection keys (excluding live DB binaries)
rsync -a --no-perms --no-owner --no-group \
    --exclude='db-data/' --exclude='pgdata/' --exclude='postgres-data/' \
    --exclude='redisdata/' --exclude='config/data/db/' \
    "$USER_HOME/homelab/stacks/" \
    "$BACKUP_ROOT/homelab-stacks/"

logger -t "$LOG_TAG" "Nightly homelab-gpu backup completed successfully!"
EOF
sudo chmod +x /usr/local/bin/homelab-gpu-backup.sh

sudo tee /etc/systemd/system/homelab-gpu-backup.service > /dev/null << 'EOF'
[Unit]
Description=Nightly Homelab-GPU Backup (Paperless, Mail-Archiver, UniFi) to UNAS 2
After=network-online.target docker.service
Wants=network-online.target

[Service]
Type=oneshot
ExecStart=/usr/local/bin/homelab-gpu-backup.sh
EOF

sudo tee /etc/systemd/system/homelab-gpu-backup.timer > /dev/null << 'EOF'
[Unit]
Description=Run Homelab-GPU Backup Daily at 03:30 AM

[Timer]
OnCalendar=*-*-* 03:30:00
RandomizedDelaySec=10m
Persistent=true

[Install]
WantedBy=timers.target
EOF

sudo systemctl daemon-reload
sudo systemctl enable --now homelab-gpu-backup.timer
sudo systemctl start homelab-gpu-backup.service
journalctl -u homelab-gpu-backup.service -n 15 --no-pager
ls -lh /mnt/unas_backups/homelab-gpu/*/
```

### 🖼️ Task 6.1.2 Verification: Verified Multi-Node Database Dumps & Mirrors on `UNAS 2`

**Verification Objective:** Capture a screenshot showing your automated dumps (`immich_db_*.sql.gz`, `paperless_db_*.sql.gz`, `mailarchiver_db_*.sql.gz`) safely written inside `/mnt/unas_backups/` on your `UNAS 2`.

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification screenshot below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

### Task 6.1.3: Optional — Attach Migrated Photos on `UNAS 2` as a Read-Only Immich External Library
Did your University OneDrive contain years of campus photos, lab project pictures, or DSLR shoots? Rather than copying gigabytes of legacy photos onto `homelab-pi`'s NVMe SSD, mount `/mnt/unas_media` **read-only** into `immich-server`:

1. On `homelab-pi` (`ssh homelab-pi`), add the read-only bind mount to `~/homelab/stacks/immich/docker-compose.yml` under `immich-server.volumes`:
   ```yaml
   services:
     immich-server:
       volumes:
         - ${UPLOAD_LOCATION}:/usr/src/app/upload
         - /etc/localtime:/etc/localtime:ro
         # Read-only UNAS 2 RAID 1 External Library:
         - /mnt/unas_media:/mnt/unas_media:ro
   ```
2. Recreate the container cleanly (`cd ~/homelab/stacks/immich && docker compose up -d`).
3. In the **Immich Web UI (`http://10.0.0.200:2283`) $\rightarrow$ Administration $\rightarrow$ External Libraries $\rightarrow$ Create External Library**, point the import path to `/mnt/unas_media/University_OneDrive_Archive/Photos` (or whichever subfolder holds your archival photos). Immich will index every legacy photo using `homelab-gpu`'s NVIDIA CUDA worker while keeping the master files untouched and read-only on `UNAS 2`!

## 🏁 Module Checkpoint & Reflection

Before proceeding to the final capstone guide, confirm the following checklist:

- [ ] `homelab-backup.timer` (`03:00 AM`) active on `homelab-pi` dumping `immich_postgres` and mirroring originals + configs to `UNAS 2`.
- [ ] `homelab-gpu-backup.timer` (`03:30 AM`) active on `homelab-gpu` dumping `paperless_db`, `mailarchiver_postgres`, UniFi `.unf` backups, and Compose stacks to `UNAS 2`.
- [ ] Manual test runs verified `.sql.gz` dumps present inside `/mnt/unas_backups/`.
- [ ] Verification screenshot captured and attached above.

Proceed to [Module 6.2: Zero-Knowledge Encrypted 5 TB Google Drive Cloud Backups (`rclone crypt`) & Full-Cluster Monitoring](./06_2_automated_5tb_google_drive_cloud_backup_and_monitoring.ipynb).